In [ ]:
import torch

__all__ = ["PreFetcher"]


class PreFetcher:
    r""" Data pre-fetcher to accelerate the data loading
    """

    # 初始方法
    def __init__(self, loader, ismulti):
        # 原始数据加载器
        self.ori_loader = loader
        # 数据批次数量
        self.len = len(loader)
        # 创建专用CUDA流，用于数据预取
        self.stream = torch.cuda.Stream()
        # 初始化预取数据存储，用于存储预取的下一个批次数据
        self.next_input = None
        # 多模态标志
        self.ismulti = ismulti

    # 数据预取方法
    def preload(self):


        try:
            # 获取下一批数据
            self.next_input = next(self.loader)
        except StopIteration:
            self.next_input = None
            return

        # 在专用流中执行数据传输
        with torch.cuda.stream(self.stream):
            for idx, tensor in enumerate(self.next_input):
                # 非阻塞方式传输到GPU，异步传输，不阻塞当前CPU线程
                self.next_input[idx] = tensor.cuda(non_blocking=True)

    # 返回数据长度
    def __len__(self):
        return self.len

    # 迭代器初始化
    def __iter__(self):
        self.loader = iter(self.ori_loader)
        self.preload()
        return self

    # 获取下一批数据
    def __next__(self):
        # 等待预取流完成
        torch.cuda.current_stream().wait_stream(self.stream)
        # 获取预取的数据
        input = self.next_input

        # 迭代终止条件
        if input is None:
            raise StopIteration

        # 记录张量使用的流
        for idx, tensor in enumerate(input):
            tensor.record_stream(torch.cuda.current_stream())
        # 立即开始预取下一批
        self.preload()

        # 返回当前批次
        return input